# Error propagation

A calculated result is only as certain as the measurements it is calculated from. Working out how the
uncertainties of the measurements carry over to the result is usually called *error propagation*. As the
note in the arithmetic notebook explains, the number it produces is an **uncertainty** (a statement about
how well a value is known), so that is the word used here.

The result of a calculation depends on the measurements **and on the formula** that combines them. The same
quantity can often be worked out in several ways, and the ways differ in how well they use the
measurements. This notebook uses one example throughout: the area of a table top.

For independent inputs the uncertainty of $f(x_1, x_2, \dots)$ follows, to first order, from

$$
\sigma_f^2 = \sum_i \left(\frac{\partial f}{\partial x_i}\right)^2 \sigma_i^2 .
$$

The term $\partial f / \partial x_i \cdot \sigma_i$ is the **contribution** of measurement $i$: its uncertainty,
scaled by how strongly the result reacts to it. A formula that reacts weakly to uncertain measurements
gives a small uncertainty.

In [1]:
import numpy as np
import plotly.graph_objects as go

import easyscience as es
from easyscience import Parameter

The function `show(label, number)` prints a number as `value ± uncertainty unit` and, for a `Parameter` with finite bounds, the bounds. The uncertainty is the attribute `error`; see the note in the arithmetic notebook.

In [2]:
def show(label, number):
    """Print a labelled number as value ± uncertainty with its unit and, for a Parameter, bounds."""
    text = f'{label} = {number.value:.4f} ± {number.error or 0.0:.4f} {number.unit}'
    if hasattr(number, 'min') and (np.isfinite(number.min) or np.isfinite(number.max)):
        text += f'   in [{number.min + 0.0:.4f}, {number.max + 0.0:.4f}]'
    print(text)

In [3]:
import base64
import contextlib
import html
import io
import re

from IPython import get_ipython
from IPython.display import HTML
from IPython.display import IFrame
from IPython.display import display
from pygments import highlight
from pygments.formatters import HtmlFormatter
from pygments.lexers import PythonLexer


def show_figure(fig: go.Figure, height: int = 420) -> None:
    """Show a Plotly figure as an embedded iframe, so that it renders in the docs."""
    page = (
        go
        .Figure(fig)
        .update_layout(template='plotly_white')
        .to_html(
            full_html=True,
            include_plotlyjs='cdn',
            default_width='100%',
            default_height=f'{height - 20}px',
        )
    )
    encoded = base64.b64encode(page.encode('utf-8')).decode('ascii')
    display(IFrame(src=f'data:text/html;base64,{encoded}', width='100%', height=height))


def solution(note: str, source: str) -> None:
    """Run a solution in a copy of the notebook namespace and show it, folded, with its output."""
    code = source.strip('\n')
    namespace = dict(get_ipython().user_ns)
    stream = io.StringIO()
    with contextlib.redirect_stdout(stream):
        exec(code, namespace)
    parts = [
        '<details class="example"><summary>Solution</summary>',
        highlight(code, PythonLexer(), HtmlFormatter()),
    ]
    result = stream.getvalue().rstrip('\n')
    if result:
        parts.append('<p>Output:</p><pre>' + html.escape(result) + '</pre>')
    if note.strip():
        text = re.sub('`([^`]+)`', r'<code>\1</code>', html.escape(note.strip()))
        parts.append('<p>' + text + '</p>')
    parts.append('</details>')
    display(HTML(''.join(parts)))


get_ipython().register_magic_function(solution, magic_kind='cell')

## Three ways to find the area of a table

The top of a table is a rectangle, 2.00 m long and 1.50 m wide, so its area is 3.00 m². Suppose that you
cannot measure the area directly. Every measurement with the tape measure has a standard uncertainty of
±0.01 m. You can measure the length $L$, the width $W$, the diagonal $D$ and the perimeter $P$, and the area follows from two of them in
three different ways:

1. from the length and the width: $A = L\,W$,
2. from the length and the perimeter, because the width is $P/2 - L$: $A = L\,(P/2 - L)$,
3. from the length and the diagonal, with Pythagoras: $A = L\,\sqrt{D^2 - L^2}$.

All three give 3.00 m². Their uncertainties are different.

In [4]:
length = Parameter(name='length', value=2.00, unit='m', variance=0.01**2)
width = Parameter(name='width', value=1.50, unit='m', variance=0.01**2)
diagonal = Parameter(name='diagonal', value=2.50, unit='m', variance=0.01**2)
perimeter = Parameter(name='perimeter', value=7.00, unit='m', variance=0.01**2)

### Method 1: length times width

EasyScience does the work. Each measurement is used once, which is the case where its result is correct (to first order):

In [5]:
area_1 = length * width
show('area_1', area_1)

area_1 = 3.0000 ± 0.0250 m^2


By hand, $\sigma^2 = (W\,\sigma_L)^2 + (L\,\sigma_W)^2 = (1.50 \cdot 0.01)^2 + (2.00 \cdot 0.01)^2\ \text{m}^4
= 0.000625\ \text{m}^4$, so $\sigma = 0.0250\ \text{m}^2$.

### A helper for any formula

The other two formulas use a measurement twice, and the result of EasyScience is not correct for those (the next
sections show why). To compare all three on equal terms, this notebook uses two short helpers. `gradient`
finds the derivatives $\partial f / \partial x_i$ numerically and `propagate` combines them with the
uncertainties exactly as in the formula above. Together with `report`, which prints a result like `show`
does, they are plain Python and work for any formula:

In [6]:
def gradient(func, values):
    """Derivatives of func with respect to each value, by central differences."""
    values = np.asarray(values, dtype=float)
    slopes = np.empty_like(values)
    for i, value in enumerate(values):
        step = 1e-6 * max(abs(value), 1.0)
        upper, lower = values.copy(), values.copy()
        upper[i] += step
        lower[i] -= step
        slopes[i] = (func(*upper) - func(*lower)) / (2 * step)
    return slopes


def propagate(func, values, sigmas, correlation=None):
    """First-order propagation: the value of func and its uncertainty."""
    sigmas = np.asarray(sigmas, dtype=float)
    if correlation is None:
        correlation = np.eye(len(sigmas))
    covariance = np.asarray(correlation, dtype=float) * np.outer(sigmas, sigmas)
    slopes = gradient(func, values)
    return func(*values), float(np.sqrt(slopes @ covariance @ slopes))


def report(label, result, unit):
    """Print a labelled (value, uncertainty) pair like show does."""
    print(f'{label} = {result[0]:.4f} ± {result[1]:.4f} {unit}')

For the first formula the helper must agree with EasyScience, which it does:

In [7]:
report('by_helper', propagate(lambda L, W: L * W, [2.00, 1.50], [0.01, 0.01]), 'm^2')
show('area_1', area_1)

by_helper = 3.0000 ± 0.0250 m^2
area_1 = 3.0000 ± 0.0250 m^2


### Methods 2 and 3, and why the formula matters

The three methods side by side, each with the same tape-measure uncertainty of 0.01 m on every
measurement. For each method the contribution of every measurement, $\partial A / \partial x_i \cdot
\sigma_i$, is printed too:

In [8]:
sigma = 0.01
methods = [
    ('length and width', ['length', 'width'], lambda L, W: L * W, [2.00, 1.50]),
    ('length and perimeter', ['length', 'perimeter'], lambda L, P: L * (P / 2 - L), [2.00, 7.00]),
    (
        'length and diagonal',
        ['length', 'diagonal'],
        lambda L, D: L * np.sqrt(D**2 - L**2),
        [2.00, 2.50],
    ),
]

for label, names, func, values in methods:
    area, uncertainty = propagate(func, values, [sigma, sigma])
    slopes = gradient(func, values)
    parts = ', '.join(f'{name} {slope * sigma:+.4f}' for name, slope in zip(names, slopes))
    print(f'{label:<22} area = {area:.4f} ± {uncertainty:.4f} m^2   ({parts})')

length and width       area = 3.0000 ± 0.0250 m^2   (length +0.0150, width +0.0200)
length and perimeter   area = 3.0000 ± 0.0112 m^2   (length -0.0050, perimeter +0.0100)
length and diagonal    area = 3.0000 ± 0.0353 m^2   (length -0.0117, diagonal +0.0333)


Measuring the length and the perimeter gives the smallest uncertainty, 0.011 m², less than half of the 0.025 m²
of the obvious length-times-width. In $L\,(P/2 - L)$ a longer length also means a shorter width, so the two
uses of $L$ almost cancel: the derivative with respect to $L$ is only $P/2 - 2L = -0.5$, and the
uncertainty of $L$ hardly matters. The perimeter, on the other hand, enters weakly (with the factor
$L/2$). The diagonal is the worst choice: $\partial A / \partial D = L D / \sqrt{D^2 - L^2} = 3.33$ makes
the uncertainty of the diagonal three times as important as in a plain product.

In [9]:
labels = [label for label, *_ in methods]
uncertainties = [propagate(func, values, [sigma, sigma])[1] for _, _, func, values in methods]

fig = go.Figure(
    go.Bar(
        x=labels,
        y=uncertainties,
        marker_color='rgb(99, 110, 250)',
        hovertemplate='%{x}<br>uncertainty = %{y:.4f} m²<extra></extra>',
    )
)
fig.update_layout(
    xaxis_title='measured quantities',
    yaxis_title='uncertainty of the area / m²',
    margin=dict(l=60, r=20, t=30, b=50),
)
show_figure(fig)

The same measurements, with the same uncertainty each, give a result that is more or less certain depending only on the formula.

### Exercise 1: A round table

A round table top has a diameter of 1.20 ± 0.01 m. Its circumference was also measured, 3.77 ± 0.01 m. Work
out the area both ways, from the diameter, $A = \pi d^2 / 4$, and from the circumference,
$A = C^2 / (4\pi)$. Each measurement is used once, so the result of EasyScience is correct. Which gives the smaller
uncertainty?

In [10]:
diameter = Parameter(name='diameter', value=1.20, unit='m', variance=0.01**2)
circumference = Parameter(name='circumference', value=3.77, unit='m', variance=0.01**2)

<details class="question">
<summary>Hint 1</summary>

The first area is `np.pi * diameter**2 / 4`. A plain number can multiply or divide a `Parameter`.

</details>

<details class="question">
<summary>Hint 2</summary>

The relative uncertainty of $x^2$ is twice that of $x$. Compare $0.01 / 1.20$ with $0.01 / 3.77$.

</details>

In [11]:
%%solution The circumference gives the smaller uncertainty because 0.01 m is a smaller fraction of 3.77 m than of 1.20 m.
area_from_diameter = np.pi * diameter**2 / 4
area_from_circumference = circumference**2 / (4 * np.pi)
show('area_from_diameter', area_from_diameter)
show('area_from_circumference', area_from_circumference)

### Exercise 2: Which measurement matters most?

A block measures 0.50 m, 0.20 m and 0.10 m, each with an uncertainty of 0.01 m. Work out its volume with its
uncertainty, then print the contribution $|\partial V / \partial x_i| \cdot \sigma_i$ of each side, in m³.
Which side dominates?

In [12]:
block_length = Parameter(name='block_length', value=0.50, unit='m', variance=0.01**2)
block_width = Parameter(name='block_width', value=0.20, unit='m', variance=0.01**2)
block_thickness = Parameter(name='block_thickness', value=0.10, unit='m', variance=0.01**2)

<details class="question">
<summary>Hint 1</summary>

The volume is the product of the three sides, so the result of EasyScience is correct.

</details>

<details class="question">
<summary>Hint 2</summary>

The contributions need the derivatives of $abc$. `gradient(lambda a, b, c: a * b * c, [0.50, 0.20, 0.10])` returns them in the order of the sides.

</details>

In [13]:
%%solution The thinnest side dominates, although all three have the same uncertainty, because 0.01 m is 10 % of 0.10 m but only 2 % of 0.50 m.
volume = block_length * block_width * block_thickness
show('volume', volume)
slopes = gradient(lambda a, b, c: a * b * c, [0.50, 0.20, 0.10])
for name, slope in zip(['length', 'width', 'thickness'], slopes):
    print(f'{name} contributes {abs(slope) * 0.01:.5f} m^3')

## When a measurement is used twice

Arithmetic in EasyScience knows nothing about where its operands come from. In $L\,(P/2 - L)$ the factor $P/2 - L$ is a
**new** number, and the multiplication by $L$ does not know that it contains $L$. It treats
the two as independent, which they are not. For the table:

In [14]:
by_operators = length * (perimeter / 2 - length)
show('by_operators', by_operators)
report('by_helper', propagate(lambda L, P: L * (P / 2 - L), [2.00, 7.00], [0.01, 0.01]), 'm^2')

by_operators = 3.0000 ± 0.0269 m^2
by_helper = 3.0000 ± 0.0112 m^2


EasyScience gives an uncertainty more than twice as large as the correct one. The same happens with
any formula in which a measurement appears more than once, and it can go either way. A small example with a
dimensionless $x = 3.0 \pm 0.1$ and $f = x\,(1 + x)$, whose derivative is $1 + 2x = 7$ and whose uncertainty
should therefore be $0.7$:

In [15]:
x = Parameter(name='x', value=3.0, variance=0.1**2)
show('x * (1 + x)', x * (1 + x))
report('by_helper', propagate(lambda x: x * (1 + x), [3.0], [0.1]), 'dimensionless')

x * (1 + x) = 12.0000 ± 0.5000 dimensionless
by_helper = 12.0000 ± 0.7000 dimensionless


Three things help:

* Where a formula uses each measurement **once**, the result of EasyScience is correct. Prefer such a formula if there
  is one, as in the round table.
* Using the *same object* on both sides of one operation is fine: `x * x`, `x**2` and `x - x` all have the right
  uncertainty.
* Otherwise work the derivatives out, by hand or with a helper such as `propagate` above.

### Exercise 3: Length and perimeter

A rectangular plate is 0.80 ± 0.01 m long and has a perimeter of 2.60 ± 0.01 m. Its area is
$A = L\,(P/2 - L)$. Work out the area with EasyScience and with `propagate`. How different are the two
uncertainties?

In [16]:
plate_length = Parameter(name='plate_length', value=0.80, unit='m', variance=0.01**2)
plate_perimeter = Parameter(name='plate_perimeter', value=2.60, unit='m', variance=0.01**2)

<details class="question">
<summary>Hint 1</summary>

The length appears twice in the formula, so EasyScience treats the two as independent.

</details>

<details class="question">
<summary>Hint 2</summary>

`propagate(lambda L, P: L * (P / 2 - L), [0.80, 2.60], [0.01, 0.01])` returns the value and the uncertainty. `report` prints them.

</details>

In [17]:
%%solution EasyScience overestimates the uncertainty by a factor of two.
by_operators = plate_length * (plate_perimeter / 2 - plate_length)
show('by_operators', by_operators)
report('by_helper', propagate(lambda L, P: L * (P / 2 - L), [0.80, 2.60], [0.01, 0.01]), 'm^2')

## Choosing the procedure

Not only the formula but also the way of measuring decides the uncertainty. The period of a pendulum of
length $\ell$ gives the acceleration of gravity, $g = 4\pi^2 \ell / T^2$. Because $T$ is squared, its
relative uncertainty counts twice, and a timing uncertainty of 0.1 s on one swing of 2 s is 5 %. If the
time of ten swings is measured instead and divided by ten, the same 0.1 s of timing uncertainty is spread over
ten swings.

### Exercise 4: A pendulum

A pendulum of length 1.000 ± 0.002 m is timed. One swing took 2.0 ± 0.1 s, and ten swings together took
20.0 ± 0.1 s. Work out $g$ both ways, in m/s². How much does timing ten swings help?

In [18]:
pendulum_length = Parameter(name='pendulum_length', value=1.000, unit='m', variance=0.002**2)
one_swing = Parameter(name='one_swing', value=2.0, unit='s', variance=0.1**2)
ten_swings = Parameter(name='ten_swings', value=20.0, unit='s', variance=0.1**2)

<details class="question">
<summary>Hint 1</summary>

The period of the second measurement is `ten_swings / 10`. Dividing by a plain number divides the uncertainty too.

</details>

<details class="question">
<summary>Hint 2</summary>

Each measurement is used once in $4\pi^2 \ell / T^2$, so the result of EasyScience is correct.

</details>

In [19]:
%%solution Timing ten swings makes the uncertainty of g about ten times smaller.
g_one_swing = 4 * np.pi**2 * pendulum_length / one_swing**2
period_ten = ten_swings / 10
g_ten_swings = 4 * np.pi**2 * pendulum_length / period_ten**2
show('g_one_swing', g_one_swing)
show('g_ten_swings', g_ten_swings)

## Good to know

EasyScience combines numbers with the arithmetic operators and the functions of `es.math`, and these see only the numbers they are given, not where those numbers
came from. Three consequences are worth knowing, because a wrong result looks just as confident as a right
one.

**The same object in one operation is recognised; the same quantity reached by two routes is not.** `x * x`
is right, but `x * x - x**2` and `(x + 1) - x` combine intermediate results that no longer remember `x`.
Each line gives the correct uncertainty and the one that EasyScience reports:

In [20]:
x = Parameter(name='x', value=3.0, variance=0.1**2)

same_object = x * x
two_routes = x * x - x**2
shifted = (x + 1) - x

print(f'x * x         correct {2 * 3.0 * 0.1:.4f}   EasyScience {same_object.error:.4f}')
print(f'x * x - x**2  correct {0.0:.4f}   EasyScience {two_routes.error:.4f}')
print(f'(x + 1) - x   correct {0.0:.4f}   EasyScience {shifted.error:.4f}')

x * x         correct 0.6000   EasyScience 0.6000
x * x - x**2  correct 0.0000   EasyScience 0.8485
(x + 1) - x   correct 0.0000   EasyScience 0.1414


The first uncertainty is exact. The other two quantities are exactly $0$ and $1$, with no uncertainty at all,
but EasyScience reports a noticeable one.

**The functions of `es.math` behave in the same way.** The identity $\sin^2\theta + \cos^2\theta = 1$ holds
exactly, but the two terms are treated as independent:

In [21]:
angle = Parameter(name='angle', value=0.7, unit='rad', variance=0.05**2)
identity = es.math.sin(angle) ** 2 + es.math.cos(angle) ** 2

print(f'sin² + cos²   correct {0.0:.4f}   EasyScience {identity.error:.4f}')

sin² + cos²   correct 0.0000   EasyScience 0.0697


**The bounds of a `Parameter` are combined in the same way.** Take a number between 0 and 10. Its difference
with itself is always 0, but EasyScience assumes that the two sides vary independently:

In [22]:
bounded = Parameter(name='bounded', value=4.0, variance=0.1**2, min=0.0, max=10.0)
difference = bounded - bounded

print(
    f'bounded - bounded   correct bounds [0.0, 0.0]   EasyScience [{difference.min}, {difference.max}]'
)

bounded - bounded   correct bounds [0.0, 0.0]   EasyScience [-10.0, 10.0]


`bounded - bounded` is a special case that EasyScience recognises, so the result is exact for the
uncertainty. Its bounds are still those of two independent numbers, $[0 - 10, 10 - 0]$.

When a formula uses a measurement more than once, the safest course is the one in this notebook: check the
result against a short calculation with the derivatives, as `propagate` does.